### Library imports

In [ ]:
from pathlib import Path
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt; plt.style.use('default')
import pandas as pd
from collections import defaultdict
import pickle

In [ ]:
# path to softmax scores for each subject  for expected format)
data_dir = Path('./phi35')

In [ ]:
# 5, 10, 15, 20

In [ ]:
# List of expected files
expected_files = [
    'computer_security_scores.npy',
    'computer_security_targets.npy',
    'high_school_computer_science_scores.npy',
    'high_school_computer_science_targets.npy',
    'college_computer_science_scores.npy',
    'college_computer_science_targets.npy',
    'machine_learning_scores.npy',
    'machine_learning_targets.npy',
    'formal_logic_scores.npy',
    'formal_logic_targets.npy',
    'high_school_biology_scores.npy',
    'high_school_biology_targets.npy',
    'anatomy_scores.npy',
    'anatomy_targets.npy',
    'clinical_knowledge_scores.npy',
    'clinical_knowledge_targets.npy',
    'college_medicine_scores.npy',
    'college_medicine_targets.npy',
    'professional_medicine_scores.npy',
    'professional_medicine_targets.npy',
    'college_chemistry_scores.npy',
    'college_chemistry_targets.npy',
    'marketing_scores.npy',
    'marketing_targets.npy',
    'public_relations_scores.npy',
    'public_relations_targets.npy',
    'management_scores.npy',
    'management_targets.npy',
    'business_ethics_scores.npy',
    'business_ethics_targets.npy',
    'professional_accounting_scores.npy',
    'professional_accounting_targets.npy',
]

# Check for missing files
missing_files = [file for file in expected_files if not (data_dir / file).exists()]

if missing_files:
    print("The following files are missing:")
    for file in missing_files:
        print(file)
else:
    print("All files are present.")

In [ ]:
# Load softmax scores and correct answers for each multiple choice question 

datasets = {
    'computer security': {
        'scores': np.load(data_dir / 'computer_security_scores.npy')[0],
        'targets': np.load(data_dir / 'computer_security_targets.npy'),
    },
    'high school computer science': {
        'scores': np.load(data_dir / 'high_school_computer_science_scores.npy')[0],
        'targets': np.load(data_dir / 'high_school_computer_science_targets.npy'),
    },
    'college computer science': {
        'scores': np.load(data_dir / 'college_computer_science_scores.npy')[0],
        'targets': np.load(data_dir / 'college_computer_science_targets.npy'),
    },
    'machine learning': {
        'scores': np.load(data_dir / 'machine_learning_scores.npy')[0],
        'targets': np.load(data_dir / 'machine_learning_targets.npy'),
    },
    'formal logic': {
        'scores': np.load(data_dir / 'formal_logic_scores.npy')[0],
        'targets': np.load(data_dir / 'formal_logic_targets.npy'),
    },
    'high school biology': {
        'scores': np.load(data_dir / 'high_school_biology_scores.npy')[0],
        'targets': np.load(data_dir / 'high_school_biology_targets.npy'),
    },
    'anatomy': {
        'scores': np.load(data_dir / 'anatomy_scores.npy')[0],
        'targets': np.load(data_dir / 'anatomy_targets.npy'),
    },
    'clinical knowledge': {
        'scores': np.load(data_dir / 'clinical_knowledge_scores.npy')[0],
        'targets': np.load(data_dir / 'clinical_knowledge_targets.npy'),
    },
    'college medicine': {
        'scores': np.load(data_dir / 'college_medicine_scores.npy')[0],
        'targets': np.load(data_dir / 'college_medicine_targets.npy'),
    },
    'professional medicine': {
        'scores': np.load(data_dir / 'professional_medicine_scores.npy')[0],
        'targets': np.load(data_dir / 'professional_medicine_targets.npy'),
    },
    'college chemistry': {
        'scores': np.load(data_dir / 'college_chemistry_scores.npy')[0],
        'targets': np.load(data_dir / 'college_chemistry_targets.npy'),
    },
    'marketing': {
        'scores': np.load(data_dir / 'marketing_scores.npy')[0],
        'targets': np.load(data_dir / 'marketing_targets.npy'),
    },
    'public relations': {
        'scores': np.load(data_dir / 'public_relations_scores.npy')[0],
        'targets': np.load(data_dir / 'public_relations_targets.npy'),
    },
    'management': {
        'scores': np.load(data_dir / 'management_scores.npy')[0],
        'targets': np.load(data_dir / 'management_targets.npy'),
    },
    'business ethics': {
        'scores': np.load(data_dir / 'business_ethics_scores.npy')[0],
        'targets': np.load(data_dir / 'business_ethics_targets.npy'),
    },
    'professional accounting': {
        'scores': np.load(data_dir / 'professional_accounting_scores.npy')[0],
        'targets': np.load(data_dir / 'professional_accounting_targets.npy'),
    },
}

# Experiments

In [ ]:
alpha = 0.05  # Desired error rate
num_trials = 100  # Number of random trials to perform

In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm import tqdm
import math
from decimal import Decimal, ROUND_HALF_UP

def round_half_up(n):
    return int(Decimal(n).to_integral_value(rounding=ROUND_HALF_UP))

def calculate_r_value(datasets, alpha, num_trials, num_rephrasings):
    all_r_value_sets = defaultdict(list)
    all_test_targets = defaultdict(list)
    all_sizes_rvalue = defaultdict(list)
    all_average_sizes = defaultdict(list)
    all_correct_predictions = defaultdict(list)

    for i in tqdm(range(num_trials)):
        for name, dataset in datasets.items():
            probabilities = dataset['scores'][:, :num_rephrasings, :]
            targets = dataset['targets']

            # Shuffle data
            index = np.arange(len(probabilities))
            np.random.shuffle(index)

            probabilities = probabilities[index]
            targets = targets[index]

            # Split data into validation and test sets
            validation_probabilities = probabilities[:len(probabilities) // 2]
            validation_targets = targets[:len(targets) // 2]
            test_probabilities = probabilities[len(probabilities) // 2:]
            test_targets = targets[len(targets) // 2:]

            validation_correct_probs = np.array([validation_probabilities[i, :, t] for i, t in enumerate(validation_targets)])
            val_true_probs = validation_correct_probs.T

            r_value_sets = []
            for test_sample in test_probabilities:
                all_probs = np.concatenate([val_true_probs, test_sample], axis=1)
                sorted_indices = np.argsort(-all_probs, axis=1)

                num_rows, num_cols = sorted_indices.shape
                index_counts = np.zeros(num_cols, dtype=int)
                is_used = np.zeros(num_cols, dtype=bool)

                results = []
                for col in range(num_cols):
                    np.add.at(index_counts, sorted_indices[:, col], 1)
                    masked_counts = np.where(is_used, -1, index_counts)
                    max_idx = np.argmax(masked_counts)
                    r_value = (col + 1) / num_cols
                    results.append((max_idx, r_value))
                    is_used[max_idx] = True

                results_df = pd.DataFrame(results, columns=['index', 'r_value'])
                test = results_df[results_df['index'] >= len(validation_correct_probs)]
                val = results_df[results_df['index'] < len(validation_correct_probs)]

                # boundary = val.iloc[round((1-alpha)*len(validation_correct_probs))]['r_value']
                # boundary = val.iloc[round_half_up((1-alpha)*len(validation_correct_probs))]['r_value']
                # boundary = val.iloc[math.ceil((1 - alpha) * len(validation_correct_probs))]['r_value']
                boundary = val.iloc[math.ceil((len(validation_correct_probs) + 1) * (1 - alpha))]['r_value']
                # print(f'Boundary: {(1 - alpha) * len(validation_correct_probs)}')
                test_filtered = test[test['r_value'] <= boundary]
                test_filtered.loc[:, 'index'] = test_filtered['index'] - len(validation_correct_probs)
                r_value_sets.append({
                    'alpha': alpha,
                    'data': {key: list(value.values()) for key, value in test_filtered.to_dict().items()}
                })

            all_average_sizes[name].append(np.mean([len(r['data']['index']) for r in r_value_sets]))
            
            # Collect results
            correct_predictions = sum(1 for i, r in enumerate(r_value_sets)
                                    if test_targets[i] in r['data']['index']) / len(test_targets)

            all_r_value_sets[name].append(r_value_sets)
            all_test_targets[name].append(test_targets)
            all_sizes_rvalue[name].append([len(r['data']['index']) for r in r_value_sets])
            all_correct_predictions[name].append(correct_predictions)
            
    # Report results
    print(f'R-VALUE COVERAGE at alpha: {alpha}')
    print()
    for name, results in all_correct_predictions.items():
        print(name.center(50, '-'))
        print(f'Coverage: {np.mean(results):.2%} +/- {np.std(results):.2%}')
        print()

    print('********************')
    print(f'SET SIZES at alpha: {alpha}')
    print()
    for name, results in all_average_sizes.items():
        print(name.center(50, '-'))
        print(f'Set Size: {np.mean(results):.1f} +/- {np.std(results):.1f}')
        print()
        
    return all_average_sizes, all_correct_predictions

In [ ]:
all_sizes_rvalue, all_correct_predictions = calculate_r_value(datasets, alpha, num_trials, num_rephrasings=21)

In [ ]:
models = ['llama1b', 'llama3b', 'mistral7b', 'phi35', 'qwen7b']
all_results = []
for model in models:
    data_dir = Path(f'./{model}')
    results_dict = {
        'num_rephrasings': [],
        'set_size_mean': [],
        'set_size_std': [],
        'coverage_mean': [],
        'coverage_std': [],
    }
    
    datasets = {
        'computer security': {
            'scores': np.load(data_dir / 'computer_security_scores.npy')[0],
            'targets': np.load(data_dir / 'computer_security_targets.npy'),
        },
        'high school computer science': {
            'scores': np.load(data_dir / 'high_school_computer_science_scores.npy')[0],
            'targets': np.load(data_dir / 'high_school_computer_science_targets.npy'),
        },
        'college computer science': {
            'scores': np.load(data_dir / 'college_computer_science_scores.npy')[0],
            'targets': np.load(data_dir / 'college_computer_science_targets.npy'),
        },
        'machine learning': {
            'scores': np.load(data_dir / 'machine_learning_scores.npy')[0],
            'targets': np.load(data_dir / 'machine_learning_targets.npy'),
        },
        'formal logic': {
            'scores': np.load(data_dir / 'formal_logic_scores.npy')[0],
            'targets': np.load(data_dir / 'formal_logic_targets.npy'),
        },
        'high school biology': {
            'scores': np.load(data_dir / 'high_school_biology_scores.npy')[0],
            'targets': np.load(data_dir / 'high_school_biology_targets.npy'),
        },
        'anatomy': {
            'scores': np.load(data_dir / 'anatomy_scores.npy')[0],
            'targets': np.load(data_dir / 'anatomy_targets.npy'),
        },
        'clinical knowledge': {
            'scores': np.load(data_dir / 'clinical_knowledge_scores.npy')[0],
            'targets': np.load(data_dir / 'clinical_knowledge_targets.npy'),
        },
        'college medicine': {
            'scores': np.load(data_dir / 'college_medicine_scores.npy')[0],
            'targets': np.load(data_dir / 'college_medicine_targets.npy'),
        },
        'professional medicine': {
            'scores': np.load(data_dir / 'professional_medicine_scores.npy')[0],
            'targets': np.load(data_dir / 'professional_medicine_targets.npy'),
        },
        'college chemistry': {
            'scores': np.load(data_dir / 'college_chemistry_scores.npy')[0],
            'targets': np.load(data_dir / 'college_chemistry_targets.npy'),
        },
        'marketing': {
            'scores': np.load(data_dir / 'marketing_scores.npy')[0],
            'targets': np.load(data_dir / 'marketing_targets.npy'),
        },
        'public relations': {
            'scores': np.load(data_dir / 'public_relations_scores.npy')[0],
            'targets': np.load(data_dir / 'public_relations_targets.npy'),
        },
        'management': {
            'scores': np.load(data_dir / 'management_scores.npy')[0],
            'targets': np.load(data_dir / 'management_targets.npy'),
        },
        'business ethics': {
            'scores': np.load(data_dir / 'business_ethics_scores.npy')[0],
            'targets': np.load(data_dir / 'business_ethics_targets.npy'),
        },
        'professional accounting': {
            'scores': np.load(data_dir / 'professional_accounting_scores.npy')[0],
            'targets': np.load(data_dir / 'professional_accounting_targets.npy'),
        },
    }
    
    alpha = 0.05  # Desired error rate
    num_trials = 10  # Number of random trials to perform   
    
        
    for num_rephrasings in [5, 10, 15, 20]:
        
        
        
        all_sizes_rvalue, all_correct_predictions = calculate_r_value(datasets, alpha, num_trials, num_rephrasings)
        all_average_coverage = defaultdict(list)
        for name in datasets.keys():
            all_average_coverage[name].append(np.mean(all_correct_predictions[name]))
            all_average_coverage[name].append(np.std(all_correct_predictions[name]))
            
        # now average across all datasets and get std
        all_average_coverage_all = defaultdict(list)
        for name in datasets.keys():
            all_average_coverage_all["using_rvalue_mean"].append(all_average_coverage[name][0])
            all_average_coverage_all["using_rvalue_std"].append(all_average_coverage[name][1])
            
            
        for name in ["using_rvalue_mean", "using_rvalue_std"]:
            all_average_coverage_all[name] = np.mean(all_average_coverage_all[name])
        
        all_average_sizes = defaultdict(list)
        for name in datasets.keys():
            all_average_sizes[name].append(np.mean(all_sizes_rvalue[name]))
            all_average_sizes[name].append(np.std(all_sizes_rvalue[name]))
            
        all_average_sizes_all = defaultdict(list)
        for name in datasets.keys():
            all_average_sizes_all["using_rvalue_mean"].append(all_average_sizes[name][0])
            all_average_sizes_all["using_rvalue_std"].append(all_average_sizes[name][1])

        for name in ["using_rvalue_mean", "using_rvalue_std"]:
            all_average_sizes_all[name] = np.mean(all_average_sizes_all[name])
        
        # apend results in a dictionary
        results_dict['num_rephrasings'].append(num_rephrasings)
        results_dict['set_size_mean'].append(all_average_sizes_all["using_rvalue_mean"])
        results_dict['set_size_std'].append(all_average_sizes_all["using_rvalue_std"])
        results_dict['coverage_mean'].append(all_average_coverage_all["using_rvalue_mean"])
        results_dict['coverage_std'].append(all_average_coverage_all["using_rvalue_std"])
    
    all_results.append(results_dict)

In [ ]:
import plotly.graph_objects as go

# Example models and corresponding results (replace all_results with your computed data)
models = ['llama1b', 'llama3b', 'mistral7b', 'phi35', 'qwen7b']
# all_results should be a list of dictionaries, one per model, as in your code snippet.

# --- Plot for Set Size Mean vs. Number of Rephrasings ---
fig_set_size = go.Figure()

# Add a trace for each model (initially set all traces as not visible)
for i, (model, results) in enumerate(zip(models, all_results)):
    fig_set_size.add_trace(
        go.Scatter(
            x=results['num_rephrasings'],
            y=results['set_size_mean'],
            error_y=dict(type='data', array=results['set_size_std']),
            mode='lines+markers',
            name=model,
            visible=False  # we'll control visibility with the dropdown
        )
    )

# Make the first model visible initially
fig_set_size.data[0].visible = True

# Create dropdown buttons to update visible trace
buttons = []
for i, model in enumerate(models):
    # Create a list to set visibility: only the current model's trace is True
    visibility = [False] * len(models)
    visibility[i] = True
    button = dict(
        label=model,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Set Size vs. Number of Rephrasings for {model}"}]
    )
    buttons.append(button)

fig_set_size.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=1.15,  # position to the right of the plot
        xanchor="right",
        y=1.2,
        yanchor="top"
    )],
    title=f"Set Size vs. Number of Rephrasings for {models[0]}",
    xaxis_title="Number of Rephrasings",
    yaxis_title="Set Size Mean"
)

fig_set_size.show()

# --- Plot for Coverage Mean vs. Number of Rephrasings ---
fig_coverage = go.Figure()

for i, (model, results) in enumerate(zip(models, all_results)):
    fig_coverage.add_trace(
        go.Scatter(
            x=results['num_rephrasings'],
            y=results['coverage_mean'],
            error_y=dict(type='data', array=results['coverage_std']),
            mode='lines+markers',
            name=model,
            visible=False
        )
    )

# Make the first model visible initially
fig_coverage.data[0].visible = True

buttons = []
for i, model in enumerate(models):
    visibility = [False] * len(models)
    visibility[i] = True
    button = dict(
        label=model,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Coverage vs. Number of Rephrasings for {model}"}]
    )
    buttons.append(button)

fig_coverage.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=1.15,
        xanchor="right",
        y=1.2,
        yanchor="top"
    )],
    title=f"Coverage vs. Number of Rephrasings for {models[0]}",
    xaxis_title="Number of Rephrasings",
    yaxis_title="Coverage Mean"
)

fig_coverage.show()

In [ ]:
from functools import partial
import numpy as np
import torch


def calibrate_lac(scores, targets, alpha=0.1, return_dist=False):
    """
    Estimates the 1-alpha quantile on held-out calibration data.
    The score function is `1 - max(softmax_score)`.
    
    Arguments:
        scores: softmax scores of the calibration set
        targets: corresponding labels of the calibration set
        alpha: parameter for the desired coverage level (1-alpha)

    Returns:
       qhat: the estimated quantile
       score_dist: the score distribution
    """
    scores = torch.tensor(scores, dtype=torch.float)
    targets = torch.tensor(targets)
    assert scores.size(0) == targets.size(0)
    assert targets.size(0)
    n = torch.tensor(targets.size(0))
    assert n

    score_dist = torch.take_along_dim(1 - scores, targets.unsqueeze(1), 1).flatten()
    assert (
        0 <= torch.ceil((n + 1) * (1 - alpha)) / n <= 1
    ), f"{alpha=} {n=} {torch.ceil((n+1)*(1-alpha))/n=}"
    qhat = torch.quantile(
        
        score_dist, torch.ceil((n + 1) * (1 - alpha)) / n, interpolation="higher"
    )
    return (qhat, score_dist) if return_dist else qhat


def inference_lac(scores, qhat, allow_empty_sets=False):
    """
    Makes prediction sets on new test data
    
    Arguments:
        scores: softmax scores of the test set
        qhat: estimated quantile of the calibration set from the `calirbate_lac` function
        allow_empty_sets: if True allow a prediction set to contain no predictions (will then satisfy upper bound of marginal coverage)

    Returns:
       prediction_sets: boolean mask of prediction sets (True if class is included in the prediction set; otherwise False)
    """
    scores = torch.tensor(scores, dtype=torch.float)
    n = scores.size(0)

    elements_mask = scores >= (1 - qhat)

    if not allow_empty_sets:
        elements_mask[torch.arange(n), scores.argmax(1)] = True
        
    prediction_sets = elements_mask

    return prediction_sets



def get_coverage(psets, targets, precision=None):
    """
    Calculates empirical coverage of prediction sets
    
    Arguments:
        psets: prediction sets of test set
        targets: ground true labels of test set
        precision: rounding precision

    Returns:
       coverage: how many times the answer is in the prediction set
    """
    psets = torch.tensor(psets)
    targets = torch.tensor(targets)
    psets = psets.clone()
    targets = targets.clone()
    n = psets.shape[0]
    coverage = psets[torch.arange(n), targets].float().mean().item()
    # if precision is not None:
    #     coverage = round(coverage, precision)
    return coverage


def get_size(psets, precision=1):
    """
    Calculates empirical set sizes of prediction sets (can consider as the average uncertainty of the model)
    
    Arguments:
        psets: prediction sets of test set
        precision: rounding precision

    Returns:
       size: how many prediction does each set contain on average
    """
    psets = psets.clone()
    size = psets.sum(1).float().mean().item()
    # if precision is not None:
    #     size = round(size, precision)
    return size

In [ ]:
from collections import defaultdict
def get_regular_conformal(datasets, alpha, num_trials):
    all_q_hats = defaultdict(list)
    all_scores = defaultdict(list)
    all_psets = defaultdict(list)
    all_targets = defaultdict(list)
    all_coverage = defaultdict(list)
    all_size = defaultdict(list)

    for i in range(num_trials):
        for name, results in datasets.items():
            scores = results['scores'][:, 0, :]  
            # scores = results['scores'].mean(axis=1)  # new: average over all 21 rephrasings
            targets = results['targets']
            
            # shuffle data
            index = np.arange(len(scores))
            np.random.shuffle(index)

            # split data into calibration and test sets
            n = len(scores) // 2
            cal_scores = scores[index][:n]
            val_scores = scores[index][n:]
            cal_targets = targets[index][:n]
            val_targets = targets[index][n:]

            # estimate 1-alpha quantile on calibration set
            q = calibrate_lac(cal_scores, cal_targets, alpha=alpha)

            # make prediction sets on test set
            psets = inference_lac(val_scores, q)
            
            all_psets[name].append(psets)
            all_scores[name].append(val_scores)
            all_targets[name].append(val_targets)
            
            # Calculate coverage and set size for this trial
            coverage = get_coverage(psets, val_targets)
            size = get_size(psets)
            
            all_coverage[name].append(coverage)
            all_size[name].append(size)

    print(f'COVERAGE at alpha: {alpha}')
    print()

    for name in datasets.keys():
        print(name.center(50, '-'))
        mean_coverage = np.mean(all_coverage[name])
        std_coverage = np.std(all_coverage[name])
        print(f'{mean_coverage:.0%} +/- {std_coverage:.0%}')
        print()

    print('********************')
    print(f'SET SIZES at alpha: {alpha}')
    print()

    for name in datasets.keys():
        print(name.center(50, '-'))
        mean_size = np.mean(all_size[name])
        std_size = np.std(all_size[name])
        print(f'{mean_size:.1f} +/- {std_size:.1f}')
        print()
    
    return all_coverage, all_size

In [ ]:
from collections import defaultdict
def get_mean_conformal(datasets, alpha, num_trials):
    all_q_hats = defaultdict(list)
    all_scores = defaultdict(list)
    all_psets = defaultdict(list)
    all_targets = defaultdict(list)
    all_coverage_using_mean = defaultdict(list)
    all_size_using_mean = defaultdict(list)

    for i in range(num_trials):
        for name, results in datasets.items():
            # scores = results['scores'][:, 0, :]  
            scores = results['scores'].mean(axis=1)  # new: average over all 21 rephrasings
            targets = results['targets']
            
            # shuffle data
            index = np.arange(len(scores))
            np.random.shuffle(index)

            # split data into calibration and test sets
            n = len(scores) // 2
            cal_scores = scores[index][:n]
            val_scores = scores[index][n:]
            cal_targets = targets[index][:n]
            val_targets = targets[index][n:]

            # estimate 1-alpha quantile on calibration set
            q = calibrate_lac(cal_scores, cal_targets, alpha=alpha)

            # make prediction sets on test set
            psets = inference_lac(val_scores, q)
            
            all_psets[name].append(psets)
            all_scores[name].append(val_scores)
            all_targets[name].append(val_targets)
            
            # Calculate coverage and set size for this trial
            coverage = get_coverage(psets, val_targets)
            size = get_size(psets)
            
            all_coverage_using_mean[name].append(coverage)
            all_size_using_mean[name].append(size)

    print(f'COVERAGE at alpha: {alpha}')
    print()

    for name in datasets.keys():
        print(name.center(50, '-'))
        mean_coverage = np.mean(all_coverage_using_mean[name])
        std_coverage = np.std(all_coverage_using_mean[name])
        print(f'{mean_coverage:.0%} +/- {std_coverage:.0%}')
        print()

    print('********************')
    print(f'SET SIZES at alpha: {alpha}')
    print()

    for name in datasets.keys():
        print(name.center(50, '-'))
        mean_size = np.mean(all_size_using_mean[name])
        std_size = np.std(all_size_using_mean[name])
        print(f'{mean_size:.1f} +/- {std_size:.1f}')
        print()
    
    return all_coverage_using_mean, all_size_using_mean

In [ ]:
# same for coverage
all_average_coverage = defaultdict(list)
for name in datasets.keys():
    all_average_coverage[name].append(np.mean(all_coverage_using_mean[name]))
    all_average_coverage[name].append(np.mean(all_coverage[name]))
    all_average_coverage[name].append(np.mean(all_correct_predictions[name]))
    
# now average across all datasets
all_average_coverage_all = defaultdict(list)
for name in datasets.keys():
    all_average_coverage_all["using_mean"].append(all_average_coverage[name][0])
    all_average_coverage_all["regular_cp"].append(all_average_coverage[name][1])
    all_average_coverage_all["using_rvalue"].append(all_average_coverage[name][2])
    
for name in ["using_mean", "regular_cp", "using_rvalue"]:
    all_average_coverage_all[name] = np.mean(all_average_coverage_all[name])
all_average_coverage_all

In [ ]:
# get the average set size for all datasets for all_size_using_mean, all_size, all_sizes_rvalue
# these are three different methods of estimating the set size
# then average across all datasets
all_average_sizes = defaultdict(list)
for name in datasets.keys():
    all_average_sizes[name].append(np.mean(all_size_using_mean[name]))
    all_average_sizes[name].append(np.mean(all_size[name]))
    all_average_sizes[name].append(np.mean(all_sizes_rvalue[name]))
    
# now average across all datasets
all_average_sizes_all = defaultdict(list)
for name in datasets.keys():
    all_average_sizes_all["using_mean"].append(all_average_sizes[name][0])
    all_average_sizes_all["regular_cp"].append(all_average_sizes[name][1])
    all_average_sizes_all["using_rvalue"].append(all_average_sizes[name][2])

for name in ["using_mean", "regular_cp", "using_rvalue"]:
    all_average_sizes_all[name] = np.mean(all_average_sizes_all[name])
all_average_sizes_all
    

In [ ]:
models = ['llama1b', 'llama3b', 'mistral7b', 'phi35', 'qwen7b']
all_results = []

results_dict = {
    'models': models,
    'set_size_std_regular': [],
    'coverage_std_regular': [],
    'set_size_std_mean': [],
    'coverage_std_mean': [],
    'set_size_std_r': [],
    'coverage_std_r': [],
    'set_size_mean_regular' : [],
    'coverage_mean_regular' : [],
    'set_size_mean_mean' : [],
    'coverage_mean_mean' : [],
    'set_size_mean_r': [],
    'coverage_mean_r': [],
}
for model in models:
    data_dir = Path(f'./{model}')
    
    datasets = {
        'computer security': {
            'scores': np.load(data_dir / 'computer_security_scores.npy')[0],
            'targets': np.load(data_dir / 'computer_security_targets.npy'),
        },
        'high school computer science': {
            'scores': np.load(data_dir / 'high_school_computer_science_scores.npy')[0],
            'targets': np.load(data_dir / 'high_school_computer_science_targets.npy'),
        },
        'college computer science': {
            'scores': np.load(data_dir / 'college_computer_science_scores.npy')[0],
            'targets': np.load(data_dir / 'college_computer_science_targets.npy'),
        },
        'machine learning': {
            'scores': np.load(data_dir / 'machine_learning_scores.npy')[0],
            'targets': np.load(data_dir / 'machine_learning_targets.npy'),
        },
        'formal logic': {
            'scores': np.load(data_dir / 'formal_logic_scores.npy')[0],
            'targets': np.load(data_dir / 'formal_logic_targets.npy'),
        },
        'high school biology': {
            'scores': np.load(data_dir / 'high_school_biology_scores.npy')[0],
            'targets': np.load(data_dir / 'high_school_biology_targets.npy'),
        },
        'anatomy': {
            'scores': np.load(data_dir / 'anatomy_scores.npy')[0],
            'targets': np.load(data_dir / 'anatomy_targets.npy'),
        },
        'clinical knowledge': {
            'scores': np.load(data_dir / 'clinical_knowledge_scores.npy')[0],
            'targets': np.load(data_dir / 'clinical_knowledge_targets.npy'),
        },
        'college medicine': {
            'scores': np.load(data_dir / 'college_medicine_scores.npy')[0],
            'targets': np.load(data_dir / 'college_medicine_targets.npy'),
        },
        'professional medicine': {
            'scores': np.load(data_dir / 'professional_medicine_scores.npy')[0],
            'targets': np.load(data_dir / 'professional_medicine_targets.npy'),
        },
        'college chemistry': {
            'scores': np.load(data_dir / 'college_chemistry_scores.npy')[0],
            'targets': np.load(data_dir / 'college_chemistry_targets.npy'),
        },
        'marketing': {
            'scores': np.load(data_dir / 'marketing_scores.npy')[0],
            'targets': np.load(data_dir / 'marketing_targets.npy'),
        },
        'public relations': {
            'scores': np.load(data_dir / 'public_relations_scores.npy')[0],
            'targets': np.load(data_dir / 'public_relations_targets.npy'),
        },
        'management': {
            'scores': np.load(data_dir / 'management_scores.npy')[0],
            'targets': np.load(data_dir / 'management_targets.npy'),
        },
        'business ethics': {
            'scores': np.load(data_dir / 'business_ethics_scores.npy')[0],
            'targets': np.load(data_dir / 'business_ethics_targets.npy'),
        },
        'professional accounting': {
            'scores': np.load(data_dir / 'professional_accounting_scores.npy')[0],
            'targets': np.load(data_dir / 'professional_accounting_targets.npy'),
        },
    }
    
    alpha = 0.05  # Desired error rate
    num_trials = 100  # Number of random trials to perform   

        
        
        
    all_sizes_rvalue, all_correct_predictions = calculate_r_value(datasets, alpha, num_trials, num_rephrasings=21)
    all_coverage, all_size = get_regular_conformal(datasets, alpha, num_trials)
    all_coverage_using_mean, all_size_using_mean = get_mean_conformal(datasets, alpha, num_trials)
    
    all_average_coverage = defaultdict(list)
    for name in datasets.keys():
        all_average_coverage[name].append(np.std(all_coverage_using_mean[name]))
        all_average_coverage[name].append(np.mean(all_coverage_using_mean[name]))
        all_average_coverage[name].append(np.std(all_coverage[name]))
        all_average_coverage[name].append(np.mean(all_coverage[name]))
        all_average_coverage[name].append(np.std(all_correct_predictions[name]))
        all_average_coverage[name].append(np.mean(all_correct_predictions[name]))
        
    # now average across all datasets
    all_average_coverage_all = defaultdict(list)
    for name in datasets.keys():
        all_average_coverage_all["using_mean_std"].append(all_average_coverage[name][0])
        all_average_coverage_all["using_mean_mean"].append(all_average_coverage[name][1])
        all_average_coverage_all["regular_cp_std"].append(all_average_coverage[name][2])
        all_average_coverage_all["regular_cp_mean"].append(all_average_coverage[name][3])
        all_average_coverage_all["using_rvalue_std"].append(all_average_coverage[name][4])
        all_average_coverage_all["using_rvalue_mean"].append(all_average_coverage[name][5])
        
    for name in ["using_mean_std", "using_mean_mean", "regular_cp_std", "regular_cp_mean", "using_rvalue_std", "using_rvalue_mean"]:
        all_average_coverage_all[name] = np.mean(all_average_coverage_all[name])
    
    all_average_sizes = defaultdict(list)
    for name in datasets.keys():
        all_average_sizes[name].append(np.std(all_size_using_mean[name]))
        all_average_sizes[name].append(np.mean(all_size_using_mean[name]))
        all_average_sizes[name].append(np.std(all_size[name]))
        all_average_sizes[name].append(np.mean(all_size[name]))
        all_average_sizes[name].append(np.std(all_sizes_rvalue[name]))
        all_average_sizes[name].append(np.mean(all_sizes_rvalue[name]))
        
    # now average across all datasets
    all_average_sizes_all = defaultdict(list)
    for name in datasets.keys():
        all_average_sizes_all["using_mean_std"].append(all_average_sizes[name][0])
        all_average_sizes_all["using_mean_mean"].append(all_average_sizes[name][1])
        all_average_sizes_all["regular_cp_std"].append(all_average_sizes[name][2])
        all_average_sizes_all["regular_cp_mean"].append(all_average_sizes[name][3])
        all_average_sizes_all["using_rvalue_std"].append(all_average_sizes[name][4])
        all_average_sizes_all["using_rvalue_mean"].append(all_average_sizes[name][5])

    for name in ["using_mean_std", "using_mean_mean", "regular_cp_std", "regular_cp_mean", "using_rvalue_std", "using_rvalue_mean"]:
        all_average_sizes_all[name] = np.mean(all_average_sizes_all[name])
    
    # apend results in a dictionary
    results_dict['set_size_std_regular'].append(all_average_sizes_all["regular_cp_std"])
    results_dict['coverage_std_regular'].append(all_average_coverage_all["regular_cp_std"])
    results_dict['set_size_std_mean'].append(all_average_sizes_all["using_mean_std"])
    results_dict['coverage_std_mean'].append(all_average_coverage_all["using_mean_std"])
    results_dict['set_size_std_r'].append(all_average_sizes_all["using_rvalue_std"])
    results_dict['coverage_std_r'].append(all_average_coverage_all["using_rvalue_std"])
    results_dict['set_size_mean_r'].append(all_average_sizes_all["using_rvalue_mean"])
    results_dict['coverage_mean_r'].append(all_average_coverage_all["using_rvalue_mean"])
    results_dict['set_size_mean_regular'].append(all_average_sizes_all["regular_cp_mean"])
    results_dict['coverage_mean_regular'].append(all_average_coverage_all["regular_cp_mean"])
    results_dict['set_size_mean_mean'].append(all_average_sizes_all["using_mean_mean"])
    results_dict['coverage_mean_mean'].append(all_average_coverage_all["using_mean_mean"])
    

In [ ]:
models = ['llama1b', 'llama3b', 'mistral7b', 'phi35', 'qwen7b']
all_results = []

for model in models:
    data_dir = Path(f'./{model}')
    
    datasets = {
        'computer security': {
            'scores': np.load(data_dir / 'computer_security_scores.npy')[0],
            'targets': np.load(data_dir / 'computer_security_targets.npy'),
        },
        'high school computer science': {
            'scores': np.load(data_dir / 'high_school_computer_science_scores.npy')[0],
            'targets': np.load(data_dir / 'high_school_computer_science_targets.npy'),
        },
        'college computer science': {
            'scores': np.load(data_dir / 'college_computer_science_scores.npy')[0],
            'targets': np.load(data_dir / 'college_computer_science_targets.npy'),
        },
        'machine learning': {
            'scores': np.load(data_dir / 'machine_learning_scores.npy')[0],
            'targets': np.load(data_dir / 'machine_learning_targets.npy'),
        },
        'formal logic': {
            'scores': np.load(data_dir / 'formal_logic_scores.npy')[0],
            'targets': np.load(data_dir / 'formal_logic_targets.npy'),
        },
        'high school biology': {
            'scores': np.load(data_dir / 'high_school_biology_scores.npy')[0],
            'targets': np.load(data_dir / 'high_school_biology_targets.npy'),
        },
        'anatomy': {
            'scores': np.load(data_dir / 'anatomy_scores.npy')[0],
            'targets': np.load(data_dir / 'anatomy_targets.npy'),
        },
        'clinical knowledge': {
            'scores': np.load(data_dir / 'clinical_knowledge_scores.npy')[0],
            'targets': np.load(data_dir / 'clinical_knowledge_targets.npy'),
        },
        'college medicine': {
            'scores': np.load(data_dir / 'college_medicine_scores.npy')[0],
            'targets': np.load(data_dir / 'college_medicine_targets.npy'),
        },
        'professional medicine': {
            'scores': np.load(data_dir / 'professional_medicine_scores.npy')[0],
            'targets': np.load(data_dir / 'professional_medicine_targets.npy'),
        },
        'college chemistry': {
            'scores': np.load(data_dir / 'college_chemistry_scores.npy')[0],
            'targets': np.load(data_dir / 'college_chemistry_targets.npy'),
        },
        'marketing': {
            'scores': np.load(data_dir / 'marketing_scores.npy')[0],
            'targets': np.load(data_dir / 'marketing_targets.npy'),
        },
        'public relations': {
            'scores': np.load(data_dir / 'public_relations_scores.npy')[0],
            'targets': np.load(data_dir / 'public_relations_targets.npy'),
        },
        'management': {
            'scores': np.load(data_dir / 'management_scores.npy')[0],
            'targets': np.load(data_dir / 'management_targets.npy'),
        },
        'business ethics': {
            'scores': np.load(data_dir / 'business_ethics_scores.npy')[0],
            'targets': np.load(data_dir / 'business_ethics_targets.npy'),
        },
        'professional accounting': {
            'scores': np.load(data_dir / 'professional_accounting_scores.npy')[0],
            'targets': np.load(data_dir / 'professional_accounting_targets.npy'),
        },
    }
    
    alpha = 0.05  # Desired error rate
    num_trials = 100  # Number of random trials to perform   

        
        
        
    all_sizes_rvalue, all_correct_predictions = calculate_r_value(datasets, alpha, num_trials, num_rephrasings=21)
    all_coverage, all_size = get_regular_conformal(datasets, alpha, num_trials)
    all_coverage_using_mean, all_size_using_mean = get_mean_conformal(datasets, alpha, num_trials)
    
    lac_coverage = defaultdict(list)
    for name in datasets.keys():
        lac_coverage[name] = np.mean(all_coverage[name])
    r_value_coverage = defaultdict(list)
    for name in datasets.keys():
        r_value_coverage[name] = np.mean(all_correct_predictions[name])
    lac_set_sizes = defaultdict(list)
    for name in datasets.keys():
        lac_set_sizes[name] = np.mean(all_size[name])
    r_value_set_sizes = defaultdict(list)
    for name in datasets.keys():
        r_value_set_sizes[name] = np.mean(all_sizes_rvalue[name])
    lac_using_mean_coverage = defaultdict(list)
    for name in datasets.keys():
        lac_using_mean_coverage[name] = np.mean(all_coverage_using_mean[name])
    lac_using_mean_set_sizes = defaultdict(list)
    for name in datasets.keys():
        lac_using_mean_set_sizes[name] = np.mean(all_size_using_mean[name])
        
    # apend results in a dictionary
    results_dir = Path('./results')
    results_dir.mkdir(exist_ok=True)
    with open(results_dir / f'{model}_alpha_005_mmlu.pkl', 'wb') as f:
        pickle.dump({
            'lac_coverage': lac_coverage,
            'r_value_coverage': r_value_coverage,
            'lac_set_sizes': lac_set_sizes,
            'r_value_set_sizes': r_value_set_sizes,
            'lac_using_mean_coverage': lac_using_mean_coverage,
            'lac_using_mean_set_sizes': lac_using_mean_set_sizes,
        }, f)
    
    all_results.append({
            'lac_coverage': lac_coverage,
            'r_value_coverage': r_value_coverage,
            'lac_set_sizes': lac_set_sizes,
            'r_value_set_sizes': r_value_set_sizes,
            'lac_using_mean_coverage': lac_using_mean_coverage,
            'lac_using_mean_set_sizes': lac_using_mean_set_sizes,
        })
    

In [ ]:
import pandas as pd
from pathlib import Path

# Create a directory to save CSV files
csv_results_dir = Path('./csv_results')
csv_results_dir.mkdir(exist_ok=True)

# Iterate over the models and their corresponding results
for model, result in zip(models, all_results):
    # Create a DataFrame for coverage
    coverage_df = pd.DataFrame({
        'regular': result['lac_coverage'],
        'mean': result['lac_using_mean_coverage'],
        'R': result['r_value_coverage'],
    })
    
    # Create a DataFrame for set sizes
    set_size_df = pd.DataFrame({
        'regular': result['lac_set_sizes'],
        'mean': result['lac_using_mean_set_sizes'],
        'R': result['r_value_set_sizes'],
    })
    
    # Save the coverage DataFrame to CSV
    coverage_csv_path = csv_results_dir / f'{model}_coverage_alpha_005_mmlu.csv'
    coverage_df.to_csv(coverage_csv_path, index=True)
    
    # Save the set sizes DataFrame to CSV
    set_size_csv_path = csv_results_dir / f'{model}_set_size_alpha_005_mmlu.csv'
    set_size_df.to_csv(set_size_csv_path, index=True)

    print(f"Saved CSV files for {model}:")
    print(f" - Coverage: {coverage_csv_path}")
    print(f" - Set sizes: {set_size_csv_path}")

In [ ]:
lac_coverage = defaultdict(list)
for name in datasets.keys():
    lac_coverage[name] = np.mean(all_correct_predictions[name])
r_value_coverage = defaultdict(list)
for name in datasets.keys():
    r_value_coverage[name] = np.mean(all_coverage[name])
lac_set_sizes = defaultdict(list)
for name in datasets.keys():
    lac_set_sizes[name] = np.mean(all_size[name])
r_value_set_sizes = defaultdict(list)
for name in datasets.keys():
    r_value_set_sizes[name] = np.mean(all_average_sizes[name])
    

# lac_coverage = [np.mean(all_coverage[name]) for name in datasets.keys()]
# r_value_coverage = [np.mean(all_correct_predictions[name]) for name in datasets.keys()]

# lac_set_sizes = [np.mean(all_size[name]) for name in datasets.keys()]
# r_value_set_sizes = [np.mean(all_average_sizes[name]) for name in datasets.keys()]

# save the results
model = 'qwen_gpqa'
alpha = alpha
results_dir = Path('./results')
results_dir.mkdir(exist_ok=True)
with open(results_dir / f'{model}_alpha_{alpha}.pkl', 'wb') as f:
    pickle.dump({
        'lac_coverage': lac_coverage,
        'r_value_coverage': r_value_coverage,
        'lac_set_sizes': lac_set_sizes,
        'r_value_set_sizes': r_value_set_sizes,
    }, f)

In [ ]:
results_dict

In [ ]:
# save results
with open('./for_plot/mistral7b_coverage_lac_01.pkl', 'wb') as f:
    pickle.dump(all_coverage, f)
    
with open('./for_plot/mistral7b_sizes_lac_01.pkl', 'wb') as f:
    pickle.dump(all_size, f)

with open('./for_plot/mistral7b_coverage_r_01.pkl', 'wb') as f:
    pickle.dump(all_correct_predictions, f)

with open('./for_plot/mistral7b_sizes_r_01.pkl', 'wb') as f:
    pickle.dump(all_average_sizes, f)

In [ ]:
# plot the coverage of the R-value method and the LAC method for each dataset
plt.figure(figsize=(10, 5))
plt.plot([np.mean(all_coverage[name]) for name in datasets.keys()], label='LAC')
plt.plot([np.mean(all_correct_predictions[name]) for name in all_correct_predictions.keys()], label='R-value')
plt.xticks(range(len(datasets)), datasets.keys(), rotation=45)
plt.ylabel('Coverage')
plt.legend()
plt.title('Coverage of LAC and R-value methods')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot([np.mean(all_coverage[name]) for name in datasets.keys()], label='LAC')
plt.plot([np.mean(all_correct_predictions[name]) for name in all_correct_predictions.keys()], label='R-value')
plt.xticks(range(len(datasets)), datasets.keys(), rotation=45)
plt.ylabel('Coverage')
plt.legend()
plt.title('Coverage of LAC and R-value methods')
plt.tight_layout()
plt.show()

In [ ]:
# plot the set sizes of the R-value method and the LAC method for each dataset
plt.figure(figsize=(10, 5))
plt.plot([np.mean(all_size[name]) for name in datasets.keys()], label='LAC')
plt.plot([np.mean(all_average_sizes[name]) for name in all_average_sizes.keys()], label='R-value')
plt.xticks(range(len(datasets)), datasets.keys(), rotation=45)
plt.ylabel('Average Set Size')
plt.legend()
plt.title('Average Set Size of LAC and R-value methods')
plt.tight_layout()
plt.show()

In [ ]:
# plot the set sizes of the R-value method and the LAC method for each dataset
plt.figure(figsize=(10, 5))
plt.plot([np.mean(all_size[name]) for name in datasets.keys()], label='LAC')
plt.plot([np.mean(all_average_sizes[name]) for name in all_average_sizes.keys()], label='R-value')
plt.xticks(range(len(datasets)), datasets.keys(), rotation=45)
plt.ylabel('Average Set Size')
plt.legend()
plt.title('Average Set Size of LAC and R-value methods')
plt.tight_layout()
plt.show()

In [ ]:
lac_coverage = [np.mean(all_coverage[name]) for name in datasets.keys()]
r_value_coverage = [np.mean(all_correct_predictions[name]) for name in datasets.keys()]

lac_coverage_1 = [np.mean(all_coverage_1[name]) for name in datasets.keys()]
r_value_coverage_1 = [np.mean(all_correct_predictions_1[name]) for name in datasets.keys()]

In [ ]:
lac_size = [np.mean(all_size[name]) for name in datasets.keys()]
r_value_size = [np.mean(all_average_sizes[name]) for name in datasets.keys()]

lac_size_1 = [np.mean(all_size_1[name]) for name in datasets.keys()]
r_value_size_1 = [np.mean(all_average_sizes_1[name]) for name in datasets.keys()]

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Convert datasets.keys() to a list for compatibility
dataset_list = list(datasets.keys())

# Create a combined figure with two subplots
combined_fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.1,
    # subplot_titles=("Alpha = 0.05", "Alpha = 0.1")
)

# Add the first plot to the combined figure
combined_fig.add_trace(go.Scatter(
    x=lac_coverage,
    y=dataset_list,
    mode='markers',
    marker=dict(size=13, color='blue', symbol='circle'),
    name='Regular Conformal',
    legendgroup='Regular Conformal'  # Group legend entries
), row=1, col=1)

combined_fig.add_trace(go.Scatter(
    x=r_value_coverage,
    y=dataset_list,
    mode='markers',
    marker=dict(size=13, color='orange', symbol='triangle-up'),
    name='Ours',
    legendgroup='Ours'  # Group legend entries
), row=1, col=1)

# Add the second plot to the combined figure
combined_fig.add_trace(go.Scatter(
    x=lac_coverage_1,
    y=dataset_list,
    mode='markers',
    marker=dict(size=13, color='blue', symbol='circle'),
    name='Regular Conformal',
    legendgroup='Regular Conformal',  # Group legend entries
    showlegend=False  # Hide duplicate legend entry
), row=1, col=2)

combined_fig.add_trace(go.Scatter(
    x=r_value_coverage_1,
    y=dataset_list,
    mode='markers',
    marker=dict(size=13, color='orange', symbol='triangle-up'),
    name='Ours',
    legendgroup='Ours',  # Group legend entries
    showlegend=False  # Hide duplicate legend entry
), row=1, col=2)

# Add vertical lines to each subplot
combined_fig.add_shape(type="line",
                       x0=0.95, x1=0.95, y0=-0.5, y1=len(dataset_list) - 0.5,
                       line=dict(color="red", width=2, dash="dash"), row=1, col=1)

combined_fig.add_shape(type="line",
                       x0=0.9, x1=0.9, y0=-0.5, y1=len(dataset_list) - 0.5,
                       line=dict(color="red", width=2, dash="dash"), row=1, col=2)

# Update layout for the combined plot
combined_fig.update_layout(
    xaxis=dict(title="", range=[0.92, 1]),
    xaxis2=dict(title="", range=[0.87, 0.95]),
    yaxis=dict(tickfont=dict(size=16)),
    legend=dict(title="", orientation="h", y=-0.1, x=0.5, xanchor="center", font=dict(size=16)),
    annotations=[
        dict(
            text="Coverage",
            x=0.5,
            y=-0.08,
            showarrow=False,
            xref="paper",
            yref="paper",
            font=dict(size=16)
        ),
        dict(
            text="Alpha = 0.05",
            x=0.05,  # Adjust x position for the first title
            y=1.05,   # Position above the plot
            showarrow=False,
            xref="paper",
            yref="paper",
            font=dict(size=18, color="black")
        ),
        dict(
            text="Alpha = 0.1",
            x=0.95,  # Adjust x position for the second title
            y=1.05,   # Position above the plot
            showarrow=False,
            xref="paper",
            yref="paper",
            font=dict(size=18, color="black")
        )
    ],
    height=750, width=650, margin=dict(l=150, r=50, t=100, b=100)
)

# Show the combined plot
combined_fig.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Convert datasets.keys() to a list for compatibility
dataset_list = list(datasets.keys())

# Create a combined figure with two subplots
combined_fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.1,
    # subplot_titles=("Alpha = 0.05", "Alpha = 0.1")
)

# Add the first plot to the combined figure
combined_fig.add_trace(go.Scatter(
    x=lac_size,
    y=dataset_list,
    mode='markers',
    marker=dict(size=13, color='blue', symbol='circle'),
    name='Regular Conformal',
    legendgroup='Regular Conformal'  # Group legend entries
), row=1, col=1)

combined_fig.add_trace(go.Scatter(
    x=r_value_size,
    y=dataset_list,
    mode='markers',
    marker=dict(size=13, color='orange', symbol='triangle-up'),
    name='Ours',
    legendgroup='Ours'  # Group legend entries
), row=1, col=1)

# Add the second plot to the combined figure
combined_fig.add_trace(go.Scatter(
    x=lac_size_1,
    y=dataset_list,
    mode='markers',
    marker=dict(size=13, color='blue', symbol='circle'),
    name='Regular Conformal',
    legendgroup='Regular Conformal',  # Group legend entries
    showlegend=False  # Hide duplicate legend entry
), row=1, col=2)

combined_fig.add_trace(go.Scatter(
    x=r_value_size_1,
    y=dataset_list,
    mode='markers',
    marker=dict(size=13, color='orange', symbol='triangle-up'),
    name='Ours',
    legendgroup='Ours',  # Group legend entries
    showlegend=False  # Hide duplicate legend entry
), row=1, col=2)

# Update layout for the combined plot
combined_fig.update_layout(
    xaxis=dict(title="", range=[2.5, 4]),
    xaxis2=dict(title="", range=[2, 4]),
    yaxis=dict(tickfont=dict(size=16)),
    legend=dict(title="", orientation="h", y=-0.1, x=0.5, xanchor="center", font=dict(size=16)),
    annotations=[
        dict(
            text="Average Set Size",
            x=0.5,
            y=-0.08,
            showarrow=False,
            xref="paper",
            yref="paper",
            font=dict(size=16)
        ),
        dict(
            text="Alpha = 0.05",
            x=0.13,  # Adjust x position for the first title
            y=1.05,   # Position above the plot
            showarrow=False,
            xref="paper",
            yref="paper",
            font=dict(size=18, color="black")
        ),
        dict(
            text="Alpha = 0.1",
            x=0.87,  # Adjust x position for the second title
            y=1.05,   # Position above the plot
            showarrow=False,
            xref="paper",
            yref="paper",
            font=dict(size=18, color="black")
        )
    ],
    height=750, width=950, margin=dict(l=150, r=50, t=100, b=100)
)

# Show the combined plot
combined_fig.show()

In [ ]:
probabilities = datasets['formal logic']['scores']
targets = datasets['formal logic']['targets']

validation_probabilities = probabilities[:len(probabilities) // 2]
validation_targets = targets[:len(targets) // 2]
validation_correct_probs = np.array([validation_probabilities[i, :, t] for i, t in enumerate(validation_targets)])


In [ ]:
validation_correct_probs[:, 1:]

In [ ]:
test_probabilities = probabilities[len(probabilities) // 2:]
test_targets = targets[len(targets) // 2:]

In [ ]:
logits = datasets['college computer science']['logits']
targets = datasets['college computer science']['targets']

validation_logits = logits[:len(logits) // 2]
validation_targets = targets[:len(targets) // 2]
validation_correct_logits = np.array([validation_logits[i, :, t] for i, t in enumerate(validation_targets)])

test_logits = logits[len(logits) // 2:]
test_targets = targets[len(targets) // 2:]

In [ ]:
validation_correct_logits.shape

In [ ]:
baseline_val_logit = validation_correct_logits[:, 0]
val_logits = validation_correct_logits[:, 1:]

val_mean = np.mean(val_logits, axis=1)    
val_var = np.var(val_logits, axis=1) 

df = pd.DataFrame(baseline_val_logit)
df.to_csv(f'./for_rscript/X_val_cc.csv', index=False)

df = pd.DataFrame(val_mean)
df.to_csv(f'./for_rscript/mean_val_cc.csv', index=False)

df = pd.DataFrame(val_var)
df.to_csv(f'./for_rscript/var_val_cc.csv', index=False)

In [ ]:
val_logits.shape

In [ ]:
test_logits.shape

In [ ]:
import scipy.io
mean = np.mean(test_logits[:, 1:, :], axis=1)   
variance = np.var(test_logits[:, 1:, :], axis=1)

baseline_logits_test = test_logits[:, 0, :]

scipy.io.savemat(f'./for_rscript/test_sets_cc.mat', {'X': baseline_logits_test, 'mean': mean, 'var': variance})

In [ ]:
val_true_probs = validation_correct_probs.T

alphas = [0.1]
r_value_sets = []

for test_sample in tqdm(test_probabilities):
    all_probs = np.concatenate([val_true_probs, test_sample], axis=1)
    sorted_indices = np.argsort(-all_probs, axis=1)
    
    num_rows, num_cols = sorted_indices.shape

    index_counts = np.zeros(num_cols, dtype=int)

    # Boolean mask to track used indices
    is_used = np.zeros(num_cols, dtype=bool)

    # Results to store (index, r_value)
    results = []

    for col in range(num_cols):
        # Update frequencies for the current column in bulk
        np.add.at(index_counts, sorted_indices[:, col], 1)
        
        # Mask out used indices to find the most frequent unused index
        masked_counts = np.where(is_used, -1, index_counts)
        max_idx = np.argmax(masked_counts)
        
        # Compute r_value and store the result
        r_value = (col + 1) / num_cols
        results.append((max_idx, r_value))
        
        # Mark the selected index as used
        is_used[max_idx] = True

    results_df = pd.DataFrame(results, columns=['index', 'r_value'])

    test = results_df[results_df['index'] >= len(validation_correct_probs)]
    val = results_df[results_df['index'] < len(validation_correct_probs)]

    for alpha in alphas:
        boundry = val.iloc[int((1-alpha)*len(validation_correct_probs))]['r_value']
        test_filtered = test[test['r_value'] <= boundry]
        test_filtered['index'] = test_filtered['index'] - len(validation_correct_probs)
        r_value_sets.append({
            'alpha': alpha,
            'data': {key: list(value.values()) for key, value in test_filtered.to_dict().items()}
        })

In [ ]:
len(r_value_sets)

In [ ]:
r_value_sets[0]

In [ ]:
# find the average size of the set
average_size = np.mean([len(r['data']['index']) for r in r_value_sets])
average_size

In [ ]:
len(test_targets)

In [ ]:
len(r_value_sets)

In [ ]:
# count how many sets are empty
empty_sets = [r for r in r_value_sets if len(r['data']['index']) == 0]
len(empty_sets)

In [ ]:
# calculate the accuracy of the sets

correct = 0
for i, r in enumerate(r_value_sets):
    if test_targets[i] in r['data']['index']:
        correct += 1

correct / len(test_targets)

In [ ]:
num_trials = 100  # Number of trials
alphas = [0.1]  # Desired error rates
all_r_value_sets = defaultdict(list)
all_average_sizes = []
all_accuracies = []

for trial in range(num_trials):
    for dataset_name, dataset in datasets.items():
        probabilities = dataset['scores']
        targets = dataset['targets']

        # Shuffle data
        indices = np.arange(len(probabilities))
        np.random.shuffle(indices)

        probabilities = probabilities[indices]
        targets = targets[indices]

        # Split data into validation and test sets
        n = len(probabilities) // 2
        validation_probabilities = probabilities[:n]
        validation_targets = targets[:n]
        validation_correct_probs = np.array(
            [validation_probabilities[i, :, t] for i, t in enumerate(validation_targets)]
        )

        test_probabilities = probabilities[n:]
        test_targets = targets[n:]

        val_true_probs = validation_correct_probs.T
        r_value_sets = []

        # Perform conformal prediction for each test sample
        for test_sample in test_probabilities:
            all_probs = np.concatenate([val_true_probs, test_sample], axis=1)
            sorted_indices = np.argsort(-all_probs, axis=1)

            num_rows, num_cols = sorted_indices.shape
            index_counts = np.zeros(num_cols, dtype=int)
            is_used = np.zeros(num_cols, dtype=bool)
            results = []

            for col in range(num_cols):
                np.add.at(index_counts, sorted_indices[:, col], 1)
                masked_counts = np.where(is_used, -1, index_counts)
                max_idx = np.argmax(masked_counts)
                r_value = (col + 1) / num_cols
                results.append((max_idx, r_value))
                is_used[max_idx] = True

            results_df = pd.DataFrame(results, columns=['index', 'r_value'])
            test = results_df[results_df['index'] >= len(validation_correct_probs)]
            val = results_df[results_df['index'] < len(validation_correct_probs)]

            for alpha in alphas:
                boundary = val.iloc[int((1 - alpha) * len(validation_correct_probs))]['r_value']
                test_filtered = test[test['r_value'] <= boundary]
                test_filtered['index'] = test_filtered['index'] - len(validation_correct_probs)
                r_value_sets.append({
                    'alpha': alpha,
                    'data': {key: list(value.values()) for key, value in test_filtered.to_dict().items()}
                })

        # Aggregate size and accuracy metrics for this trial
        average_size = np.mean([len(r['data']['index']) for r in r_value_sets])
        correct = sum(
            1 for i, r in enumerate(r_value_sets)
            if test_targets[i] in r['data']['index']
        )
        accuracy = correct / len(test_targets)

        all_r_value_sets[dataset_name].append(r_value_sets)
        all_average_sizes.append(average_size)
        all_accuracies.append(accuracy)


In [ ]:
all_r_value_sets.keys()

In [ ]:
print(f'COVERAGE at alpha: {alphas[0]}')
print()

mean_coverage = {}
for dataset_name, r_value_trials in all_r_value_sets.items():
    print(dataset_name.center(50, '-'))
    
    coverages = []
    for trial_idx, r_value_set in enumerate(r_value_trials):
        correct_count = 0
        for i, r in enumerate(r_value_set):
            if test_targets[i] in r['data']['index']:
                correct_count += 1
        trial_coverage = correct_count / len(test_targets)
        coverages.append(trial_coverage)
    
    mean_coverage[dataset_name] = {
        'mean': np.mean(coverages),
        'std': np.std(coverages)
    }
    print(f"{dataset_name}: {mean_coverage[dataset_name]['mean']:.0%} +/- {mean_coverage[dataset_name]['std']:.0%}")
    print()

print('********************')
print(f'SET SIZES at alpha: {alphas[0]}')
print()

mean_size = {}
for dataset_name, r_value_trials in all_r_value_sets.items():
    print(dataset_name.center(50, '-'))
    
    sizes = []
    for r_value_set in r_value_trials:
        trial_sizes = [len(r['data']['index']) for r in r_value_set]
        sizes.append(np.mean(trial_sizes))
    
    mean_size[dataset_name] = {
        'mean': np.mean(sizes),
        'std': np.std(sizes)
    }
    print(f"{dataset_name}: {mean_size[dataset_name]['mean']:.1f} +/- {mean_size[dataset_name]['std']:.1f}")
    print()
